# Task 1 - Google Calendar Integration

This task implements the calendar-booking contract the voice agent calls once a caller
agrees to a property visit. It mirrors the shape of the real Google Calendar API
(`events.insert` / `events.patch`) so the mock client can be swapped for
`googleapiclient.discovery.build("calendar", "v3", ...)` without changing any caller code.

Each event carries the fields the client asked for — client name, phone, employee, property,
date/time, and meeting notes — plus two production requirements the task doesn't mention
explicitly but that a phone channel makes unavoidable:

- **Idempotency key** on every write. A voice call can retry a webhook after a network
  blip; the same key must return the existing event instead of creating a duplicate.
- **Conflict detection** per employee before the event is created, so two callers can't be
  booked into the same agent's slot.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone as dt_timezone
from typing import Optional
import uuid

# Employee/agent directory carried over from Day 2's structured agent records.
EMPLOYEES = {
    "AGENT-001": {"name": "Ayesha Khan", "email": "ayesha.khan@realestatehub.example"},
    "AGENT-002": {"name": "Hamza Malik", "email": "hamza.malik@realestatehub.example"},
    "AGENT-003": {"name": "Sana Ahmed", "email": "sana.ahmed@realestatehub.example"},
}

# Property directory carried over from Day 2's property records.
PROPERTIES = {
    "PROP-001": "Park View Residences",
    "PROP-002": "Gulberg Business Tower",
    "PROP-003": "DHA Garden Villas",
}


@dataclass
class CalendarEvent:
    event_id: str
    idempotency_key: str
    client_name: str
    client_phone: str
    employee_id: str
    employee_name: str
    property_id: str
    property_name: str
    start_time: datetime
    end_time: datetime
    timezone: str
    notes: str
    status: str = "confirmed"
    created_at: datetime = field(default_factory=lambda: datetime.now(dt_timezone.utc))


class GoogleCalendarConflictError(Exception):
    pass


class MockGoogleCalendarClient:
    """Simulates the subset of the Google Calendar API the agent needs:
    idempotent event creation, conflict detection, and update/cancel by event ID.
    A real implementation swaps this class body for `googleapiclient.discovery` calls
    while keeping the same method signatures.
    """

    def __init__(self) -> None:
        self._events: dict[str, CalendarEvent] = {}
        self._idempotency_index: dict[str, str] = {}  # idempotency_key -> event_id

    def _has_conflict(self, employee_id: str, start: datetime, end: datetime, exclude_event_id: Optional[str] = None) -> bool:
        for event in self._events.values():
            if event.employee_id != employee_id or event.status == "cancelled":
                continue
            if exclude_event_id and event.event_id == exclude_event_id:
                continue
            if start < event.end_time and event.start_time < end:
                return True
        return False

    def create_event(
        self,
        *,
        idempotency_key: str,
        client_name: str,
        client_phone: str,
        employee_id: str,
        property_id: str,
        start_time: datetime,
        end_time: datetime,
        notes: str,
        timezone: str = "Asia/Karachi",
    ) -> CalendarEvent:
        # Idempotency: a retried request with the same key must never create a duplicate event.
        if idempotency_key in self._idempotency_index:
            return self._events[self._idempotency_index[idempotency_key]]

        if self._has_conflict(employee_id, start_time, end_time):
            raise GoogleCalendarConflictError(
                f"{EMPLOYEES[employee_id]['name']} already has a visit booked in that slot."
            )

        event = CalendarEvent(
            event_id=f"EVT-{uuid.uuid4().hex[:8]}",
            idempotency_key=idempotency_key,
            client_name=client_name,
            client_phone=client_phone,
            employee_id=employee_id,
            employee_name=EMPLOYEES[employee_id]["name"],
            property_id=property_id,
            property_name=PROPERTIES[property_id],
            start_time=start_time,
            end_time=end_time,
            timezone=timezone,
            notes=notes,
        )
        self._events[event.event_id] = event
        self._idempotency_index[idempotency_key] = event.event_id
        return event

    def update_event_time(self, event_id: str, new_start: datetime, new_end: datetime) -> CalendarEvent:
        event = self._events[event_id]
        if self._has_conflict(event.employee_id, new_start, new_end, exclude_event_id=event_id):
            raise GoogleCalendarConflictError(
                f"{event.employee_name} already has a visit booked in that slot."
            )
        event.start_time, event.end_time = new_start, new_end
        event.status = "confirmed"
        return event

    def cancel_event(self, event_id: str) -> CalendarEvent:
        event = self._events[event_id]
        event.status = "cancelled"
        return event

    def get_event(self, event_id: str) -> CalendarEvent:
        return self._events[event_id]


calendar_client = MockGoogleCalendarClient()

# --- Book a property visit ---
visit_start = datetime(2026, 9, 29, 15, 0)
visit_end = visit_start + timedelta(minutes=45)

event = calendar_client.create_event(
    idempotency_key="visit-CLIENT-1042-PROP-001-2026-09-29T15:00",
    client_name="Bilal Farooq",
    client_phone="+92-321-5551042",
    employee_id="AGENT-001",
    property_id="PROP-001",
    start_time=visit_start,
    end_time=visit_end,
    notes="First-time buyer, wants to see the covered parking and confirm possession date.",
)
print(f"Booked {event.event_id}: {event.client_name} with {event.employee_name} "
      f"for {event.property_name} at {event.start_time:%Y-%m-%d %H:%M} {event.timezone}")

# --- Retried request with the same idempotency key must not duplicate the event ---
duplicate_attempt = calendar_client.create_event(
    idempotency_key="visit-CLIENT-1042-PROP-001-2026-09-29T15:00",
    client_name="Bilal Farooq",
    client_phone="+92-321-5551042",
    employee_id="AGENT-001",
    property_id="PROP-001",
    start_time=visit_start,
    end_time=visit_end,
    notes="duplicate webhook retry",
)
assert duplicate_attempt.event_id == event.event_id
print(f"Retry with the same idempotency key returned the existing event {duplicate_attempt.event_id} (no duplicate).")

# --- A conflicting booking for the same employee is rejected before it reaches the calendar ---
try:
    calendar_client.create_event(
        idempotency_key="visit-CLIENT-2091-PROP-001-2026-09-29T15:15",
        client_name="Noor Fatima",
        client_phone="+92-333-5552091",
        employee_id="AGENT-001",
        property_id="PROP-001",
        start_time=visit_start + timedelta(minutes=15),
        end_time=visit_start + timedelta(minutes=60),
        notes="Overlaps with Bilal Farooq's slot.",
    )
except GoogleCalendarConflictError as exc:
    print(f"Conflict correctly rejected: {exc}")

Booked EVT-355ad672: Bilal Farooq with Ayesha Khan for Park View Residences at 2026-09-29 15:00 Asia/Karachi
Retry with the same idempotency key returned the existing event EVT-355ad672 (no duplicate).
Conflict correctly rejected: Ayesha Khan already has a visit booked in that slot.


**Design notes:** `create_event` checks the idempotency index before touching the
calendar and before running conflict detection, so a retried request is a true no-op. The
conflict check runs on every write (including reschedules in Task 3) rather than only at
booking time, since a schedule change is exactly where double-booking bugs tend to hide.

# Task 2 - Email Automation

The assigned employee needs a notification the moment a visit lands on their calendar, with
enough detail to walk in prepared: meeting time, property, client contact details, and the
client's stated requirements. `render_employee_notification` builds the subject/body from a
`CalendarEvent` so the email always matches what's actually on the calendar, and
`MockEmailClient` gives the same idempotent-send guarantee as the calendar client — this
class is the swap point for a real Gmail API `users.messages.send` or Resend `emails.send`
call.

In [2]:
from dataclasses import dataclass, field
from datetime import datetime, timezone as dt_timezone
import uuid


@dataclass
class EmailMessage:
    message_id: str
    idempotency_key: str
    to_address: str
    subject: str
    body: str
    related_event_id: str
    status: str = "sent"
    sent_at: datetime = field(default_factory=lambda: datetime.now(dt_timezone.utc))


class MockEmailClient:
    """Simulates the subset of the Gmail/Resend API the agent needs: idempotent,
    templated notification emails. Swap the body of `send_email` for a real
    Gmail API `users.messages.send` or Resend `emails.send` call in production.
    """

    def __init__(self) -> None:
        self._sent: dict[str, EmailMessage] = {}
        self._idempotency_index: dict[str, str] = {}

    def send_email(self, *, idempotency_key: str, to_address: str, subject: str, body: str, related_event_id: str) -> EmailMessage:
        if idempotency_key in self._idempotency_index:
            return self._sent[self._idempotency_index[idempotency_key]]
        message = EmailMessage(
            message_id=f"MSG-{uuid.uuid4().hex[:8]}",
            idempotency_key=idempotency_key,
            to_address=to_address,
            subject=subject,
            body=body,
            related_event_id=related_event_id,
        )
        self._sent[message.message_id] = message
        self._idempotency_index[idempotency_key] = message.message_id
        return message

    def inbox_for(self, to_address: str) -> list[EmailMessage]:
        return [m for m in self._sent.values() if m.to_address == to_address]


def render_employee_notification(calendar_event, requirements: str) -> tuple[str, str]:
    """Builds the subject/body an employee receives when a visit is booked for them."""
    subject = f"New visit: {calendar_event.property_name} — {calendar_event.start_time:%a %d %b, %I:%M %p}"
    body = (
        f"Hi {calendar_event.employee_name},\n\n"
        f"A property visit has been booked on your calendar.\n\n"
        f"Property: {calendar_event.property_name} ({calendar_event.property_id})\n"
        f"Date/Time: {calendar_event.start_time:%Y-%m-%d %H:%M} - {calendar_event.end_time:%H:%M} ({calendar_event.timezone})\n"
        f"Client: {calendar_event.client_name}\n"
        f"Client phone: {calendar_event.client_phone}\n"
        f"Client requirements: {requirements}\n\n"
        f"Calendar event: {calendar_event.event_id}\n"
    )
    return subject, body


email_client = MockEmailClient()
employee_email = EMPLOYEES[event.employee_id]["email"]

subject, body = render_employee_notification(
    event, requirements="Wants covered parking confirmed and possession date in writing."
)
notification = email_client.send_email(
    idempotency_key=f"notify-{event.event_id}",
    to_address=employee_email,
    subject=subject,
    body=body,
    related_event_id=event.event_id,
)
print(f"Emailed {notification.to_address}: \"{notification.subject}\"")

# A retried notification with the same idempotency key must not send a duplicate email.
retry_notification = email_client.send_email(
    idempotency_key=f"notify-{event.event_id}",
    to_address=employee_email,
    subject=subject,
    body=body,
    related_event_id=event.event_id,
)
assert retry_notification.message_id == notification.message_id
print(f"Retry with the same idempotency key returned the existing message {retry_notification.message_id} (no duplicate email).")
print(f"{employee_email} inbox size: {len(email_client.inbox_for(employee_email))}")

Emailed ayesha.khan@realestatehub.example: "New visit: Park View Residences — Tue 29 Sep, 03:00 PM"
Retry with the same idempotency key returned the existing message MSG-686648a0 (no duplicate email).
ayesha.khan@realestatehub.example inbox size: 1


**Design notes:** the notification is generated from the calendar event object rather
than from the original call payload, so if the appointment is later rescheduled (Task 3),
regenerating the email from the updated event automatically keeps the two systems in sync
instead of relying on two independent copies of the same data.

# Task 3 - Appointment Management

`AppointmentService` is the single place that knows how to book, reschedule, and cancel a
visit, and it never lets the calendar and the employee's inbox drift apart: every state
change writes to the calendar first, then sends the corresponding email, then appends an
audit-trail entry on the `Appointment` itself. Each of the three operations is safe to
retry — a repeated booking call returns the existing appointment instead of creating a
second one, and reschedule/cancel act on the calendar's current idempotency-protected event.

In [3]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import uuid


@dataclass
class AppointmentHistoryEntry:
    action: str
    detail: str
    at: datetime


@dataclass
class Appointment:
    appointment_id: str
    calendar_event_id: str
    client_name: str
    client_phone: str
    employee_id: str
    property_id: str
    status: str  # booked | rescheduled | cancelled
    requirements: str
    history: list[AppointmentHistoryEntry] = field(default_factory=list)


class AppointmentService:
    """Coordinates the calendar and email clients so that booking, rescheduling,
    and cancellation always leave both systems — and the appointment's own audit
    trail — in a consistent state. Every public method is idempotent.
    """

    def __init__(self, calendar: MockGoogleCalendarClient, email: MockEmailClient) -> None:
        self._calendar = calendar
        self._email = email
        self._appointments: dict[str, Appointment] = {}

    def book_appointment(
        self, *, idempotency_key: str, client_name: str, client_phone: str,
        employee_id: str, property_id: str, start_time: datetime, end_time: datetime,
        requirements: str,
    ) -> Appointment:
        calendar_event = self._calendar.create_event(
            idempotency_key=idempotency_key,
            client_name=client_name,
            client_phone=client_phone,
            employee_id=employee_id,
            property_id=property_id,
            start_time=start_time,
            end_time=end_time,
            notes=requirements,
        )
        appointment_id = f"APPT-{uuid.uuid4().hex[:8]}"
        if any(a.calendar_event_id == calendar_event.event_id for a in self._appointments.values()):
            # Retry of an already-processed booking: return the existing appointment.
            return next(a for a in self._appointments.values() if a.calendar_event_id == calendar_event.event_id)

        subject, body = render_employee_notification(calendar_event, requirements)
        self._email.send_email(
            idempotency_key=f"notify-{calendar_event.event_id}-booked",
            to_address=EMPLOYEES[employee_id]["email"],
            subject=subject, body=body, related_event_id=calendar_event.event_id,
        )
        appointment = Appointment(
            appointment_id=appointment_id,
            calendar_event_id=calendar_event.event_id,
            client_name=client_name, client_phone=client_phone,
            employee_id=employee_id, property_id=property_id,
            status="booked", requirements=requirements,
        )
        appointment.history.append(AppointmentHistoryEntry(
            "booked", f"Visit booked for {calendar_event.start_time:%Y-%m-%d %H:%M}", datetime.now(dt_timezone.utc)
        ))
        self._appointments[appointment_id] = appointment
        return appointment

    def reschedule_appointment(self, appointment_id: str, new_start: datetime, new_end: datetime) -> Appointment:
        appointment = self._appointments[appointment_id]
        calendar_event = self._calendar.update_event_time(appointment.calendar_event_id, new_start, new_end)
        appointment.status = "rescheduled"
        appointment.history.append(AppointmentHistoryEntry(
            "rescheduled", f"Moved to {new_start:%Y-%m-%d %H:%M}", datetime.now(dt_timezone.utc)
        ))
        subject, body = render_employee_notification(calendar_event, appointment.requirements)
        subject = "[Rescheduled] " + subject
        self._email.send_email(
            idempotency_key=f"notify-{calendar_event.event_id}-rescheduled-{new_start.isoformat()}",
            to_address=EMPLOYEES[appointment.employee_id]["email"],
            subject=subject, body=body, related_event_id=calendar_event.event_id,
        )
        return appointment

    def cancel_appointment(self, appointment_id: str, reason: str) -> Appointment:
        appointment = self._appointments[appointment_id]
        calendar_event = self._calendar.cancel_event(appointment.calendar_event_id)
        appointment.status = "cancelled"
        appointment.history.append(AppointmentHistoryEntry("cancelled", reason, datetime.now(dt_timezone.utc)))
        self._email.send_email(
            idempotency_key=f"notify-{calendar_event.event_id}-cancelled",
            to_address=EMPLOYEES[appointment.employee_id]["email"],
            subject=f"[Cancelled] {calendar_event.property_name} visit — {calendar_event.client_name}",
            body=f"The visit for {calendar_event.client_name} on {calendar_event.start_time:%Y-%m-%d %H:%M} "
                 f"has been cancelled. Reason: {reason}",
            related_event_id=calendar_event.event_id,
        )
        return appointment


appointment_service = AppointmentService(calendar_client, email_client)

appt = appointment_service.book_appointment(
    idempotency_key="visit-CLIENT-3087-PROP-003-2026-09-30T11:00",
    client_name="Zara Sheikh", client_phone="+92-301-5553087",
    employee_id="AGENT-003", property_id="PROP-003",
    start_time=datetime(2026, 9, 30, 11, 0), end_time=datetime(2026, 9, 30, 11, 45),
    requirements="Investor client, wants square footage and payment plan details confirmed on-site.",
)
print(f"[BOOKED] {appt.appointment_id} -> calendar {appt.calendar_event_id}, status={appt.status}")

appointment_service.reschedule_appointment(appt.appointment_id, datetime(2026, 10, 1, 16, 0), datetime(2026, 10, 1, 16, 45))
print(f"[RESCHEDULED] {appt.appointment_id} -> new time "
      f"{calendar_client.get_event(appt.calendar_event_id).start_time:%Y-%m-%d %H:%M}, status={appt.status}")

appointment_service.cancel_appointment(appt.appointment_id, reason="Client found a property elsewhere.")
print(f"[CANCELLED] {appt.appointment_id} -> calendar status "
      f"{calendar_client.get_event(appt.calendar_event_id).status}, appointment status={appt.status}")

print("\nAudit trail:")
for entry in appt.history:
    print(f"  - {entry.action}: {entry.detail}")

sana_inbox = email_client.inbox_for(EMPLOYEES['AGENT-003']['email'])
print(f"\n{EMPLOYEES['AGENT-003']['name']} received {len(sana_inbox)} notifications: "
      f"{[m.subject for m in sana_inbox]}")

[BOOKED] APPT-91114998 -> calendar EVT-a2943407, status=booked
[RESCHEDULED] APPT-91114998 -> new time 2026-10-01 16:00, status=rescheduled
[CANCELLED] APPT-91114998 -> calendar status cancelled, appointment status=cancelled

Audit trail:
  - booked: Visit booked for 2026-09-30 11:00
  - rescheduled: Moved to 2026-10-01 16:00
  - cancelled: Client found a property elsewhere.

Sana Ahmed received 3 notifications: ['New visit: DHA Garden Villas — Wed 30 Sep, 11:00 AM', '[Rescheduled] New visit: DHA Garden Villas — Thu 01 Oct, 04:00 PM', '[Cancelled] DHA Garden Villas visit — Zara Sheikh']


**Design notes:** the audit trail (`Appointment.history`) is what Task 5's CRM logging
replays into `appointment_history` — the appointment object is the source of truth, and the
CRM table is a queryable projection of it, not a second place that has to be kept in sync by
hand.

# Task 4 - Workflow Automation (n8n)

```mermaid
flowchart LR
    Call([Incoming call webhook]) --> Intent[Intent classification node]
    Intent -->|success| Match[Property match node\nSQL + vector retrieval]
    Intent -->|exhausts retries| DeadA[[Dead-letter queue\n+ human escalation]]
    Match -->|success| Appt[Appointment node\nbook / reschedule / cancel]
    Match -->|exhausts retries| DeadB[[Dead-letter queue\n+ human escalation]]
    Appt -->|success| Cal[Calendar node\nGoogle Calendar API]
    Appt -->|exhausts retries| DeadC[[Dead-letter queue\n+ human escalation]]
    Cal --> Email[Email node\nnotify employee]
    Email --> CRM[CRM update node]
    CRM -->|success| Done([Workflow complete])
    CRM -->|exhausts retries| DeadD[[Dead-letter queue\n+ human escalation]]
    DeadA --> Human[Human agent follow-up queue]
    DeadB --> Human
    DeadC --> Human
    DeadD --> Human
```

Each node in the real n8n workflow is configured with n8n's built-in **retry-on-fail**
(exponential backoff, capped attempts) so a transient failure — a rate-limited Calendar
call, a slow CRM write — resolves itself without ever reaching the caller. Only a node that
exhausts its retries drops the run into a dead-letter queue and creates a human-follow-up
task; the caller-facing part of the call is never blocked waiting for a background
integration to recover.

The cell below simulates that same node graph and retry contract in Python — reusing the
`AppointmentService` from Task 3 for the appointment/calendar/email nodes — so the retry and
escalation behavior can be exercised deterministically without a live n8n instance.

In [4]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Callable
import time


@dataclass
class StepResult:
    name: str
    status: str  # success | retried | failed
    attempts: int
    detail: str = ""


@dataclass
class WorkflowRun:
    call_id: str
    steps: list[StepResult] = field(default_factory=list)
    outcome: str = "in_progress"  # completed | escalated
    dead_letter: bool = False


def with_retry(step_name: str, fn: Callable[[], str], max_attempts: int = 3, backoff_seconds: float = 0.0) -> StepResult:
    """Generic retry wrapper standing in for n8n's built-in node retry/backoff settings."""
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            detail = fn()
            status = "success" if attempt == 1 else "retried"
            return StepResult(step_name, status, attempt, detail)
        except Exception as exc:  # noqa: BLE001 - deliberately broad for a workflow-node boundary
            last_error = exc
            if attempt < max_attempts:
                time.sleep(backoff_seconds)
    return StepResult(step_name, "failed", max_attempts, str(last_error))


# In-memory CRM used as the workflow's final sink; Task 5 gives this its real, queryable form.
crm_updates: list[dict] = []
dead_letter_queue: list[dict] = []


def run_call_to_crm_workflow(
    call_payload: dict, *,
    transient_failures: set[str] | None = None,
    permanent_failures: set[str] | None = None,
) -> WorkflowRun:
    """Simulates the n8n workflow:
    Call -> Intent -> Property Match -> Appointment -> Calendar -> Email -> CRM Update
    `transient_failures` makes a node fail twice then recover on retry (the common case
    n8n's retry/backoff setting exists for). `permanent_failures` makes a node fail every
    attempt, so the run exhausts its retries and lands in the dead-letter/escalation queue.
    """
    transient_failures = transient_failures or set()
    permanent_failures = permanent_failures or set()
    run = WorkflowRun(call_id=call_payload["call_id"])
    remaining_failures = {name: 2 for name in transient_failures}

    def flaky(name: str, real_work: Callable[[], str]) -> Callable[[], str]:
        def _inner():
            if remaining_failures.get(name, 0) > 0:
                remaining_failures[name] -= 1
                raise RuntimeError(f"{name} temporarily unavailable")
            return real_work()
        return _inner

    def permanently_flaky(name: str) -> Callable[[], str]:
        def _inner():
            raise RuntimeError(f"{name} integration is down")
        return _inner

    # Node 1: Intent
    result = with_retry("intent_classification", flaky(
        "intent_classification", lambda: f"intent={call_payload['intent']}"
    ))
    run.steps.append(result)
    if result.status == "failed":
        run.outcome, run.dead_letter = "escalated", True
        dead_letter_queue.append({"call_id": run.call_id, "failed_step": result.name})
        return run

    # Node 2: Property match
    result = with_retry("property_match", flaky(
        "property_match", lambda: f"matched={call_payload['property_id']}"
    ))
    run.steps.append(result)
    if result.status == "failed":
        run.outcome, run.dead_letter = "escalated", True
        dead_letter_queue.append({"call_id": run.call_id, "failed_step": result.name})
        return run

    # Node 3: Appointment decision + booking (reuses Task 3's AppointmentService)
    def do_booking() -> str:
        appt = appointment_service.book_appointment(
            idempotency_key=f"workflow-{run.call_id}",
            client_name=call_payload["client_name"], client_phone=call_payload["client_phone"],
            employee_id=call_payload["employee_id"], property_id=call_payload["property_id"],
            start_time=call_payload["start_time"], end_time=call_payload["end_time"],
            requirements=call_payload["requirements"],
        )
        run.appointment_id = appt.appointment_id  # type: ignore[attr-defined]
        return appt.appointment_id

    fn = flaky("appointment_booking", do_booking) if "appointment_booking" in transient_failures else (
        permanently_flaky("appointment_booking") if "appointment_booking" in permanent_failures else do_booking
    )
    result = with_retry("appointment_booking", fn)
    run.steps.append(result)
    if result.status == "failed":
        run.outcome, run.dead_letter = "escalated", True
        dead_letter_queue.append({"call_id": run.call_id, "failed_step": result.name})
        return run
    # Calendar + email already happened inside book_appointment; log them as workflow nodes.
    run.steps.append(StepResult("calendar_update", "success", 1, "Google Calendar event created"))
    run.steps.append(StepResult("email_notification", "success", 1, "Employee notified by email"))

    # Node 4: CRM update
    def do_crm_update() -> str:
        crm_updates.append({"call_id": run.call_id, "appointment_id": result.detail})
        return "crm row inserted"

    fn = flaky("crm_update", do_crm_update) if "crm_update" in transient_failures else (
        permanently_flaky("crm_update") if "crm_update" in permanent_failures else do_crm_update
    )
    result = with_retry("crm_update", fn)
    run.steps.append(result)
    if result.status == "failed":
        run.outcome, run.dead_letter = "escalated", True
        dead_letter_queue.append({"call_id": run.call_id, "failed_step": result.name})
        return run

    run.outcome = "completed"
    return run


def print_run(run: WorkflowRun) -> None:
    print(f"Call {run.call_id}: outcome={run.outcome}")
    for step in run.steps:
        print(f"  - {step.name}: {step.status} (attempts={step.attempts}) {step.detail}")


# Scenario A: everything succeeds first try.
run_a = run_call_to_crm_workflow({
    "call_id": "CALL-2001", "intent": "buyer_inquiry", "property_id": "PROP-001",
    "employee_id": "AGENT-001", "client_name": "Imran Qureshi", "client_phone": "+92-315-5552001",
    "start_time": datetime(2026, 10, 3, 10, 0), "end_time": datetime(2026, 10, 3, 10, 45),
    "requirements": "Wants to see the unit before committing to the payment plan.",
})
print_run(run_a)

# Scenario B: the CRM node fails twice, then succeeds on retry.
run_b = run_call_to_crm_workflow({
    "call_id": "CALL-2002", "intent": "rental_inquiry", "property_id": "PROP-002",
    "employee_id": "AGENT-002", "client_name": "Mehwish Raza", "client_phone": "+92-333-5552002",
    "start_time": datetime(2026, 10, 3, 14, 0), "end_time": datetime(2026, 10, 3, 14, 30),
    "requirements": "Needs parking confirmed for two vehicles.",
}, transient_failures={"crm_update"})
print_run(run_b)

# Scenario C: the calendar/appointment integration is permanently down -> dead-letter + human escalation.
run_c = run_call_to_crm_workflow({
    "call_id": "CALL-2003", "intent": "buyer_inquiry", "property_id": "PROP-003",
    "employee_id": "AGENT-003", "client_name": "Talha Nadeem", "client_phone": "+92-345-5552003",
    "start_time": datetime(2026, 10, 4, 9, 0), "end_time": datetime(2026, 10, 4, 9, 45),
    "requirements": "Wants the visit this week.",
}, permanent_failures={"appointment_booking"})
print_run(run_c)

print(f"\nCRM rows written: {len(crm_updates)}")
print(f"Dead-letter / human-escalation queue: {dead_letter_queue}")

Call CALL-2001: outcome=completed
  - intent_classification: success (attempts=1) intent=buyer_inquiry
  - property_match: success (attempts=1) matched=PROP-001
  - appointment_booking: success (attempts=1) APPT-e1f6d4fb
  - calendar_update: success (attempts=1) Google Calendar event created
  - email_notification: success (attempts=1) Employee notified by email
  - crm_update: success (attempts=1) crm row inserted
Call CALL-2002: outcome=completed
  - intent_classification: success (attempts=1) intent=rental_inquiry
  - property_match: success (attempts=1) matched=PROP-002
  - appointment_booking: success (attempts=1) APPT-9cb95249
  - calendar_update: success (attempts=1) Google Calendar event created
  - email_notification: success (attempts=1) Employee notified by email
  - crm_update: retried (attempts=3) crm row inserted
Call CALL-2003: outcome=escalated
  - intent_classification: success (attempts=1) intent=buyer_inquiry
  - property_match: success (attempts=1) matched=PROP-003


**Design notes:** `CALL-2001` shows the happy path. `CALL-2002` forces the CRM node to
fail twice and recover on its third attempt — the same shape as n8n's retry-on-fail — and
still completes the workflow. `CALL-2003` forces the appointment node to fail every attempt,
which is the case a transient retry can't help with (e.g. Google Calendar credentials
revoked); the run stops, is written to the dead-letter queue, and never fabricates a
booking that doesn't exist.

# Task 5 - CRM Logging

Four tables cover what the client asked to persist: raw call transcripts, a per-client
preference profile that accumulates across calls, a full appointment history (booked /
rescheduled / cancelled), and follow-up reminders for anything that needs a human touch
later. This reuses the SQLite structured-storage pattern from Day 2's price/availability
layer, since the same argument applies here — these are operational records that need exact
lookups and joins, not semantic search.

In [5]:
import sqlite3
from datetime import datetime, timedelta
import json

crm_db = sqlite3.connect(":memory:")
crm_db.execute("""
    CREATE TABLE call_transcripts (
        call_id TEXT PRIMARY KEY,
        client_name TEXT NOT NULL,
        client_phone TEXT NOT NULL,
        transcript TEXT NOT NULL,
        intent TEXT NOT NULL,
        logged_at TEXT NOT NULL
    )
""")
crm_db.execute("""
    CREATE TABLE client_preferences (
        client_phone TEXT PRIMARY KEY,
        client_name TEXT NOT NULL,
        preferred_city TEXT,
        budget_pkr INTEGER,
        property_type TEXT,
        notes TEXT,
        updated_at TEXT NOT NULL
    )
""")
crm_db.execute("""
    CREATE TABLE appointment_history (
        history_id INTEGER PRIMARY KEY AUTOINCREMENT,
        appointment_id TEXT NOT NULL,
        action TEXT NOT NULL,
        detail TEXT NOT NULL,
        occurred_at TEXT NOT NULL
    )
""")
crm_db.execute("""
    CREATE TABLE follow_up_reminders (
        reminder_id INTEGER PRIMARY KEY AUTOINCREMENT,
        client_phone TEXT NOT NULL,
        appointment_id TEXT,
        due_at TEXT NOT NULL,
        reason TEXT NOT NULL,
        status TEXT NOT NULL DEFAULT 'pending'
    )
""")
crm_db.commit()


def log_call_transcript(call_id: str, client_name: str, client_phone: str, transcript: str, intent: str) -> None:
    crm_db.execute(
        "INSERT OR REPLACE INTO call_transcripts VALUES (?, ?, ?, ?, ?, ?)",
        (call_id, client_name, client_phone, transcript, intent, datetime.now(dt_timezone.utc).isoformat()),
    )
    crm_db.commit()


def upsert_client_preferences(client_phone: str, client_name: str, *, preferred_city: str = None,
                               budget_pkr: int = None, property_type: str = None, notes: str = "") -> None:
    crm_db.execute(
        """INSERT INTO client_preferences (client_phone, client_name, preferred_city, budget_pkr, property_type, notes, updated_at)
           VALUES (?, ?, ?, ?, ?, ?, ?)
           ON CONFLICT(client_phone) DO UPDATE SET
             preferred_city=excluded.preferred_city, budget_pkr=excluded.budget_pkr,
             property_type=excluded.property_type, notes=excluded.notes, updated_at=excluded.updated_at""",
        (client_phone, client_name, preferred_city, budget_pkr, property_type, notes, datetime.now(dt_timezone.utc).isoformat()),
    )
    crm_db.commit()


def log_appointment_event(appointment_id: str, action: str, detail: str, occurred_at: datetime) -> None:
    crm_db.execute(
        "INSERT INTO appointment_history (appointment_id, action, detail, occurred_at) VALUES (?, ?, ?, ?)",
        (appointment_id, action, detail, occurred_at.isoformat()),
    )
    crm_db.commit()


def create_follow_up_reminder(client_phone: str, appointment_id: str, due_at: datetime, reason: str) -> None:
    crm_db.execute(
        "INSERT INTO follow_up_reminders (client_phone, appointment_id, due_at, reason) VALUES (?, ?, ?, ?)",
        (client_phone, appointment_id, due_at.isoformat(), reason),
    )
    crm_db.commit()


# --- Populate from the call/appointment activity generated in Tasks 1-4 ---
log_call_transcript(
    "CALL-2001", "Imran Qureshi", "+92-315-5552001",
    transcript="Caller asked about 3-bedroom apartments in Bahria Town under 4 crore and booked a viewing.",
    intent="buyer_inquiry",
)
upsert_client_preferences(
    "+92-315-5552001", "Imran Qureshi",
    preferred_city="Lahore", budget_pkr=40000000, property_type="apartment",
    notes="Prefers Bahria Town, needs possession within 6 months.",
)

# Replay Task 3's cancelled appointment's full audit trail into the CRM.
for entry in appt.history:
    log_appointment_event(appt.appointment_id, entry.action, entry.detail, entry.at)

create_follow_up_reminder(
    appt.client_phone, appt.appointment_id,
    due_at=datetime(2026, 10, 5, 10, 0),
    reason="Client cancelled after finding another property — check if still interested in DHA listings.",
)
create_follow_up_reminder(
    "+92-315-5552001", "APPT-from-CALL-2001",
    due_at=datetime(2026, 10, 6, 10, 0),
    reason="Post-viewing follow-up call for Park View Residences.",
)

# --- Verify everything is queryable, as the CRM would need for reporting/dashboards ---
print("Call transcripts:")
for row in crm_db.execute("SELECT call_id, client_name, intent FROM call_transcripts"):
    print(" ", row)

print("\nClient preferences:")
for row in crm_db.execute("SELECT client_name, preferred_city, budget_pkr, property_type FROM client_preferences"):
    print(" ", row)

print("\nAppointment history:")
for row in crm_db.execute("SELECT appointment_id, action, detail FROM appointment_history ORDER BY history_id"):
    print(" ", row)

print("\nPending follow-up reminders:")
for row in crm_db.execute("SELECT client_phone, due_at, reason FROM follow_up_reminders WHERE status='pending' ORDER BY due_at"):
    print(" ", row)

counts = {name: crm_db.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
          for name in ("call_transcripts", "client_preferences", "appointment_history", "follow_up_reminders")}
print(f"\nRow counts: {counts}")

Call transcripts:
  ('CALL-2001', 'Imran Qureshi', 'buyer_inquiry')

Client preferences:
  ('Imran Qureshi', 'Lahore', 40000000, 'apartment')

Appointment history:
  ('APPT-91114998', 'booked', 'Visit booked for 2026-09-30 11:00')
  ('APPT-91114998', 'rescheduled', 'Moved to 2026-10-01 16:00')
  ('APPT-91114998', 'cancelled', 'Client found a property elsewhere.')

Pending follow-up reminders:
  ('+92-301-5553087', '2026-10-05T10:00:00', 'Client cancelled after finding another property — check if still interested in DHA listings.')
  ('+92-315-5552001', '2026-10-06T10:00:00', 'Post-viewing follow-up call for Park View Residences.')

Row counts: {'call_transcripts': 1, 'client_preferences': 1, 'appointment_history': 3, 'follow_up_reminders': 2}


**Design notes:** `appointment_history` is populated by replaying the `Appointment`
object's own audit trail rather than re-deriving it, so the CRM can never disagree with the
appointment it's logging. `client_preferences` uses `INSERT ... ON CONFLICT` keyed on phone
number so a returning caller's record is updated in place instead of accumulating stale
duplicate rows — the same returning-customer case the Day 1 conversation-flow design calls
out. A production version would additionally write every call/appointment event onto an
outbox table inside the same transaction as the workflow's other side effects, so a crash
between "booked the calendar" and "logged the CRM row" can be replayed rather than silently
lost.